In [1]:
import pandas as pd

df = pd.read_csv("Qwen3_0.6B.csv", encoding="utf-8")

df.head()

,viet,ref_texts,predictions,bleu,meteor,nist
0,Vượt qua những tác động tiêu cực do dịch COVID...,ដោយជំនះពុះពារលើផលប៉ះពាល់អវិជ្ជមាន ដោយសារជំងឺរា...,ជំនះពុះពារលើផលប៉ះពាល់អវិជ្ជមានដែលបណ្តាលមកពីការ...,0.671926,0.638843,0.647059
1,"Cùng với đó, các nước thành viên sẽ tăng cường...",ទន្ទឹមជាមួយគ្នានោះ បណ្ដាប្រទេសសមាជិកនឹងបង្កើនក...,រួមជាមួយគ្នានោះ បណ្ដាប្រទេសជាសមាជិកនឹងបង្កើនកិ...,0.672907,0.729036,0.804217
2,Hợp tác Mekong-Lan Thương vì một khu vực Mekon...,កិច្ចសហប្រតិបត្តិការមេគង្គ-ឡានឆាងដើម្បីតំបន់មេ...,កិច្ចសហប្រតិបត្តិការមេគង្គ - ឡាវដើម្បីតំបន់មេគ...,0.721973,0.571796,0.731032
3,"Vì vậy, phục hồi kinh tế hậu Covid-19 cũng là ...",ដូច្នេះការស្ដារសេដ្ឋកិច្ចឡើងវិញក្រោយពីកូវីដ ១៩...,ដូច្នេះ ការស្ដារសេដ្ឋកិច្ចឡើងវិញក្រោយពី Covid-...,0.734169,0.772559,0.745515
4,Điều này cho thấy sự hiện diện ngày càng mạnh ...,ប្រការនេះបង្ហាញពីវត្តមានកាន់តែខ្លាំងឡើងៗ របស់វ...,ប្រការនេះបានសរអោយឃើញថា វៀតណាមកាន់តែមានវត្តមានក...,0.675146,0.596324,0.657325


In [2]:
src_txts = []
ref_txts = []
predictions = []

for _, row in df.iterrows():
  src_txts.append(row["viet"])
  ref_txts.append(row["ref_texts"])
  # predictions.append(row["predictions"])

In [4]:
import pandas as pd

df2 = pd.read_csv("output-11.csv", encoding="utf-8")

df2.head()

,text
0,ជំនះពុះពារលើផលប៉ះពាល់អវិជ្ជមានដែលបណ្តាលមកពីការ...
1,ទន្ទឹមនឹងនោះ បណ្តាប្រទេសជាសមាជិកនឹងបង្កើនកិច្ច...
2,កិច្ចសហប្រតិបត្តិការមេគង្គ-ឡានឆាងដើម្បីតំបន់មេ...
3,ដូច្នេះ ការងើបឡើងវិញនៃសេដ្ឋកិច្ចក្រោយពី Covid-...
4,ប្រការនេះបានសរឲ្យឃើញនូវវត្តមានរបស់វៀតណាមកាន់តែ...


In [5]:

for _, row in df2.iterrows():
  predictions.append(row["text"])

In [ ]:
def translate_sailor2_batch(texts_vi, batch_size=8):
    translation_prompt = '''Dịch câu sau từ tiếng Việt sang tiếng Khmer: {}'''
    system_prompt = (
        "Bạn là chuyên gia dịch thuật chuyên về các tác vụ từ tiếng Việt sang tiếng Khmer. "
        "Hãy nhận diện các Thực thể Tên riêng (Named Entities), dịch chúng và cung cấp bản dịch hoàn chỉnh. "
        "Định dạng đầu ra: [Thực thể Nguồn] <sep> [Thực thể Đích] <sep> [Bản dịch đầy đủ]. "
        "Sử dụng '<sep>' làm dấu phân cách giữa các phần."
    )
    texts = []
    for i in range(0, len(texts_vi)):
      messages = [
          {"role":"system", "content": system_prompt},
          {"role": "user", "content": texts_vi[i]}]
      texts.append(tokenizer.apply_chat_template(
          messages, tokenize=False, add_generation_prompt=True))

    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=768
        )

    decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    results = [text.split("assistant\n")[-1].strip() for text in decoded_outputs]
    return results

In [ ]:
def batchify(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

In [ ]:
predictions = []
batch_size = 32

for i, src in enumerate(batchify(src_txts, batch_size)):
  print(i)
  pred = translate_sailor2_batch(src)
  predictions.extend(pred)

In [3]:
import re

SEP = "<sep>"

def extract_last_sep_and_remove_e(text: str) -> str:
    # 1. Lấy phần sau <sep> cuối cùng
    last_part = text.rsplit(SEP, 1)[-1].strip()

    # 2. Loại bỏ thẻ <e> </e> nhưng giữ nội dung
    clean_text = re.sub(r"</?e>", "", last_part)

    return clean_text

results = [extract_last_sep_and_remove_e(t) for t in predictions]

In [7]:
results[:5]

['ហួសពីផលប៉ះពាល់អវិជ្ជមានដែលបណ្ដាលមកពីជំងឺរាតត្បាតកូវីដ-១៩ ក្នុងរយៈពេល ៩ ខែកន្លងទៅ ប្រទេសវៀតណាមទទួលបានភាពជោគជ័យដោយទទួលបានកំណើនGDP ២,១២%។',
 'រួមជាមួយនោះ បណ្ដាប្រទេសជាសមាជិក នឹងបង្កើនកិច្ចសហប្រតិបត្តិការលើវិស័យសុខភាពដើម្បីទប់ទល់នឹងជំងឺរាតត្បាយ Covid-19; ធានាឱ្យបាននូវស្ថិរភាព សម្រាប់សកម្មភាពផលិតកម្ម និងបណ្តាញផ្គត់ផ្គង់ក្នុងតំបន់ រក្សាអត្រាកំណើនសេដ្ឋកិច្ច បន្តជំរុញការអភិវឌ្ឍន៍ប្រកបដោយចីរភាព រួមចំណែកដល់ការអនុវត្តប្រកបដោយជោគជ័យ រាល់គោលដៅ SDGs ២០៣០ ជំរុញកិច្ចសហប្រតិបត្តិការលើវិស័យអប់រំ និងការបណ្តុះបណ្តាល និងប្រភពធនធានផ្ទេរប្រភពធនធានមនុស្សសម្រាប់តំបន់មេគង្គ។',
 'កិច្ចសហប្រតិបត្តិការមេគង្គ-Lan Thuongដើម្បីតំបន់មេគង្គមួយរឹងមាំ។',
 'ដូច្នេះ ការស្តាឡើងវិញនៃសេដ្ឋកិច្ចក្រោយជំងឺ Covid-19 ក៏ជាអាទិភាពរបស់អាស៊ានផងដែរ។',
 'ប្រការនេះសរឲ្យឃើញនូវភាពម្ចាស់ការកាន់តែខ្លាំងឡើងរបស់វៀតណាមលើឆាកអន្តរជាតិ ហើយអះអាងនូវតួនាទីរបស់វៀតណាមជាអ្នកដែលមិត្តភក្តិជឿទុកចិត្ត និងជាដៃគូអន្តរជាតិផងដែរ។']

In [4]:
!pip install -q pyvi nltk khmer-nltk

In [6]:
import nltk.translate.chrf_score
from nltk.translate.meteor_score import single_meteor_score
from khmernltk import word_tokenize
import nltk
import json
import pandas as pd

import nltk.translate.nist_score
nltk.download('punkt')
nltk.download('wordnet')

from pyvi import ViTokenizer

def tokenize_vi(sentence):
    return ViTokenizer.tokenize(sentence).split()

def compute_nist(reference, translation):
    return nltk.translate.chrf_score.sentence_chrf(reference, translation)

def compute_meteor(reference, translation):
    # Use tokenized strings for METEOR score
    return single_meteor_score(reference, translation)

def compute_score(reference, translation):
    reference_tokens = word_tokenize(reference)
    translation_tokens = word_tokenize(translation)
    return compute_meteor(reference_tokens, translation_tokens), nltk.translate.bleu_score.sentence_bleu([reference], translation), compute_nist(reference_tokens, translation_tokens)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [7]:
bleu_score = 0
meteor_score = 0
bleun_score = 0
ter_score = 0
nist_score =0
id = 0
sub_scores = []
for prediction, reference in zip(predictions, ref_txts):
    m_score, bn_score, ni_score = compute_score(reference, prediction)
    # bleu_score += b_score
    meteor_score += m_score
    bleun_score += bn_score
    # ter_score += t_score
    nist_score += ni_score
    score = {
        'id': id,
        'bleu_score': bn_score,
        # 'bleu_score': b_score,
        'meteor_score': m_score,
        'chrF_score': ni_score
    }
    sub_scores.append(score)
    id += 1
result = {
    'bleu_score': bleun_score / (len(ref_txts)),
    # 'bleu_score': bleu_score / (len(prediction_data)),
    'meteor_score': meteor_score / (len(ref_txts)),
    'chrF_score': nist_score / (len(ref_txts))
}

| 2026-02-07 17:47:17,704 | INFO | khmer-nltk | Loaded model from d:\Miniconda\envs\myenv\lib\site-packages\khmernltk\word_tokenize\sklearn_crf_ner_10000.sav |


In [10]:
result

{'bleu_score': 0.6010372002259916,
 'meteor_score': 0.5425853889417566,
 'chrF_score': 0.6013836952863993}

In [9]:
result

{'bleu_score': 0.5675113454521199,
 'meteor_score': 0.49818033106689036,
 'chrF_score': 0.5640489142213014}

In [8]:
result

{'bleu_score': 0.5848829899273519,
 'meteor_score': 0.520226516485963,
 'chrF_score': 0.5824450929230461}

In [5]:
import csv

with open("test-6.txt", "r", encoding="utf-8") as txt_file, \
     open("output-6.csv", "w", newline="", encoding="utf-8") as csv_file:

    writer = csv.writer(csv_file)
    writer.writerow(["value"])  # header

    for line in txt_file:
        writer.writerow([line.strip()])


In [16]:
txt_file = "scoressss.txt"

with open(txt_file, "w", encoding="utf-8") as f:
    for idx, (src, ref, pred) in enumerate(zip(src_txts, ref_txts, predictions)):
        m_score, b_score, c_score = compute_score(ref, pred)
        if b_score < 0.35:
          f.write(f"ID: {idx}\n")
          f.write(f"SOURCE: {src}\n")
          f.write(f"REFERENCE: {ref}\n")
          f.write(f"PREDICTION: {pred}\n")
          f.write(f"BLEU: {b_score:.4f} | METEOR: {m_score:.4f} | chrF: {c_score:.4f}\n")
          f.write("="*80 + "\n\n")

print(f"✅ Done! File saved to {txt_file}")

✅ Done! File saved to scoressss.txt


In [3]:
import csv

with open("test-11.txt", "r", encoding="utf-8") as txt_file, \
     open("output-11.csv", "w", newline="", encoding="utf-8") as csv_file:

    writer = csv.writer(csv_file)

    writer.writerow(["text"])  # header

    for line in txt_file:
        writer.writerow([line.strip()])


In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# --- 1. CẤU HÌNH TRÌNH DUYỆT ---
def setup_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless") # Bỏ comment nếu muốn chạy ngầm
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--disable-notifications")
    
    # Tự động cài driver
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver

# --- 2. HÀM CRAWL DỮ LIỆU ---
def crawl_vov_lao(so_trang_muon_lay=3):
    driver = setup_driver()
    
    # URL gốc bạn cung cấp
    base_url = "https://vovworld.vn/lo-LA/%E0%BA%82%E0%BA%B2%E0%BA%A7%E0%BB%80%E0%BA%94%E0%BA%99/472.vov"
    
    all_data = []
    
    print("🚀 Bắt đầu quá trình crawl...")

    try:
        # Lặp qua từng trang (Ví dụ: Trang 1, 2, 3...)
        for page in range(701, so_trang_muon_lay + 701):
            print(f"\n=== XỬ LÝ TRANG {page} ===")
            # Tạo URL cho từng trang. VOV thường dùng cấu trúc ?trang=X
            url = f"{base_url}?trang={page}"
            
            print(f"🔄 Đang truy cập trang {page}: {url}")
            driver.get(url)
            time.sleep(3) # Đợi web tải
            
            # --- TÌM CÁC BÀI VIẾT TRÊN TRANG ---
            # Lưu ý: Class name này dựa trên cấu trúc thường thấy của VOV. 
            # Nếu web thay đổi, cần Inspect Element để cập nhật lại class này.
            articles = driver.find_elements(By.CLASS_NAME, "story") 
            
            if len(articles) == 0:
                print(f"⚠️ Không tìm thấy bài viết nào ở trang {page}. Có thể đã hết trang.")
            
            print(f"   -> Tìm thấy {len(articles)} bài viết.")

            for item in articles:
                try:
                    # 1. Lấy Tiêu đề & Link
                    # Thường nằm trong thẻ <a> có class 'story__title' hoặc thẻ <h2>/<h3>
                    title_element = item.find_element(By.CLASS_NAME, "story__heading")
                    title = title_element.text.strip() # Lấy text và xóa khoảng trắng thừa
                    
                    # 2. Lấy Thời gian (Timestamp)
                    # VOV thường để trong thẻ <time> hoặc class 'story__time' / 'time'
                    try:
                        time_element = item.find_element(By.CLASS_NAME, "story__meta")
                        published_time = time_element.text.strip() # Lấy text và xóa khoảng trắng thừa
                    except:
                        published_time = "Không có thời gian"

                    # 3. Lấy Tóm tắt (Optional)
                    try:
                        summary = item.find_element(By.CLASS_NAME, "story__summary").text
                    except:
                        summary = ""

                    # Thêm vào danh sách
                    all_data.append({
                        "Trang": page,
                        "Tiêu đề": title,
                        "Thời gian": published_time,
                        "Tóm tắt": summary
                    })
                    
                except Exception as e:
                    print(f"   ❌ Lỗi khi lấy một bài viết: {e}")
                    continue

    except Exception as e:
        print(f"❌ Có lỗi lớn xảy ra: {e}")
        
    finally:
        driver.quit()
        print("👋 Đã đóng trình duyệt.")
        
    return all_data

# --- 3. CHẠY VÀ LƯU FILE ---

# Bạn muốn lấy bao nhiêu trang thì sửa số ở đây (Ví dụ: 5 trang)
so_trang = 300
ket_qua = crawl_vov_lao(so_trang)

if ket_qua:
    df = pd.DataFrame(ket_qua)
    
    # Hiển thị kết quả mẫu
    print(f"\n✅ Tổng cộng lấy được {len(df)} bài viết.")
    display(df.head())
    
    # Lưu file Excel (Dùng utf-8-sig để không lỗi font tiếng Lào/Việt)
    df.to_csv("vov_lao_news_701_1000.csv", index=False, encoding="utf-8-sig")
    print("\n💾 Đã lưu file 'vov_lao_news_701_1000.csv'")
else:
    print("⚠️ Không lấy được dữ liệu nào.")

🚀 Bắt đầu quá trình crawl...

=== XỬ LÝ TRANG 701 ===
🔄 Đang truy cập trang 701: https://vovworld.vn/lo-LA/%E0%BA%82%E0%BA%B2%E0%BA%A7%E0%BB%80%E0%BA%94%E0%BA%99/472.vov?trang=701
   -> Tìm thấy 42 bài viết.

=== XỬ LÝ TRANG 702 ===
🔄 Đang truy cập trang 702: https://vovworld.vn/lo-LA/%E0%BA%82%E0%BA%B2%E0%BA%A7%E0%BB%80%E0%BA%94%E0%BA%99/472.vov?trang=702
   -> Tìm thấy 42 bài viết.

=== XỬ LÝ TRANG 703 ===
🔄 Đang truy cập trang 703: https://vovworld.vn/lo-LA/%E0%BA%82%E0%BA%B2%E0%BA%A7%E0%BB%80%E0%BA%94%E0%BA%99/472.vov?trang=703
   -> Tìm thấy 42 bài viết.

=== XỬ LÝ TRANG 704 ===
🔄 Đang truy cập trang 704: https://vovworld.vn/lo-LA/%E0%BA%82%E0%BA%B2%E0%BA%A7%E0%BB%80%E0%BA%94%E0%BA%99/472.vov?trang=704
   -> Tìm thấy 42 bài viết.

=== XỬ LÝ TRANG 705 ===
🔄 Đang truy cập trang 705: https://vovworld.vn/lo-LA/%E0%BA%82%E0%BA%B2%E0%BA%A7%E0%BB%80%E0%BA%94%E0%BA%99/472.vov?trang=705
   -> Tìm thấy 42 bài viết.

=== XỬ LÝ TRANG 706 ===
🔄 Đang truy cập trang 706: https://vovworld.vn/lo-L

,Trang,Tiêu đề,Thời gian,Tóm tắt
0,701,"ຜູ້ມີສິດເລືອກຕັ້ງລາວກວ່າ 4,76 ລ້ານຄົນ ເຂົ້າຮ່ວ...",Không có thời gian,
1,701,"ໄປຫາປາຢູ່ທະເລຍາມຕົ້ນປີ, ຊາວປະມົງ ແຄ້ງຮວ່າ ຕັດສ...",Không có thời gian,(VOVWORLD) - ໃນບັນຍາກາດຄືກຄື້ນມ່ວນຊືນຢ່າງຟົດຟື...
2,701,ທ່ານເລຂາທິການໃຫຍ່ ໂຕເລິມ: ປູກຕົ້ນໄມ້ໃນມື້ນີ້ເພ...,Không có thời gian,(VOVWORLD) - ການປ່ຽນແປງຂອງດິນຟ້າອາກາດນັບມື້ນັບ...
3,701,ທ່ານນາຍົກລັດຖະມົນຕີ ຟ້າມມິງຈິງ: ວາງແຜນກຳນົດກໍ່...,Không có thời gian,(VOVWORLD) - ທ່ານນາຍົກລັດຖະມົນຕີ ຟ້າມມິງຈິງ ໄດ...
4,701,ທ່ານເລຂາທິການໃຫຍ່ ຫງວຽນຝູຈ້ອງ: ເສີມຂະຫຍາຍບົດບາ...,12/01/2023,"(VOVWORLD) -ວັນທີ 12 ມັງກອນ, ຢູ່ ຮ່າໂນ້ຍ ຄະນະຊ..."



💾 Đã lưu file 'vov_lao_news_701_1000.csv'


In [2]:
import pandas as pd
import glob

folder_path = "craw/*.csv"
files = glob.glob(folder_path)

df_list = [pd.read_csv(file, encoding="utf-8-sig") for file in files]
df = pd.concat(df_list, ignore_index=True)

# Xóa cột trùng tên
df = df.loc[:, ~df.columns.duplicated()]

# 🔢 Đếm trước khi xóa
before = len(df)

# Xóa dòng trùng theo "Tiêu đề"
df = df.drop_duplicates(subset=["Tiêu đề"])

# 🔢 Đếm sau khi xóa
after = len(df)

# Số dòng bị loại bỏ
removed = before - after

print("Số dòng ban đầu:", before)
print("Số dòng sau khi loại bỏ:", after)
print("Số dòng bị loại bỏ:", removed)

df.to_csv("file_da_gop.csv", index=False)

Số dòng ban đầu: 42018
Số dòng sau khi loại bỏ: 19935
Số dòng bị loại bỏ: 22083


In [1]:
import pandas as pd
import glob

# Lấy tất cả file csv trong thư mục
files = glob.glob("csv/*.csv")

# Đọc và nối
df_list = [pd.read_csv(file) for file in files]
merged_df = pd.concat(df_list, ignore_index=True)

# Lưu file mới
merged_df.to_csv("vi_lo_10k_20k.csv", index=False)

print("Tổng số dòng sau khi nối:", len(merged_df))

Tổng số dòng sau khi nối: 9935


In [3]:
df1 = pd.read_csv("vi_lao_0_10k.csv", encoding="utf-8-sig")
df2 = pd.read_csv("vi_lo_10k_20k.csv", encoding="utf-8-sig")

merged_df = pd.concat([df1, df2], ignore_index=True)

merged_df.to_csv("vi_lo_0k_20k.csv", index=False)

print("Tổng số dòng sau khi nối:", len(merged_df))

Tổng số dòng sau khi nối: 19935
